# `klygo.files` (`metadata.py`): File metadata, sizes, hashes, and comparison

Inspect file attributes, byte sizes, checksums, and equality without changing source files.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [1]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8


In [2]:
sample_file = root / "sample.bin"
sample_file.write_bytes(b"model weights example\n")
duplicate = root / "duplicate.bin"
duplicate.write_bytes(sample_file.read_bytes())
different = root / "different.bin"
different.write_bytes(b"different bytes")
print("Sample files ready:", root)

Sample files ready: C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8


## Jump to an API

[`info`](#info) · [`size`](#size) · [`hash`](#hash) · [`compare`](#compare)

<a id="info"></a>

### `files.info`

Collect common metadata for a file or directory.

**Signature:** `files.info(path: 'PathInput', *, verbose: 'bool' = False) -> 'Dict[str, Any]'`

- `path`: Path to inspect or operate on.
- `verbose`: Print the metadata as a Rich table while returning the dictionary.

**Returns:** `dict[str, Any]` — Filesystem metadata with stable keys.

The dictionary has `name`, `stem`, `extension`, `parent`, `size`, `human_size`, `is_file`, `is_dir`, `created`, and `modified`. `parent` is a `Path`; timestamps are local `datetime` values. `verbose=True` prints a Rich table and still returns the dictionary. No checksum is computed.

In [3]:
metadata = files.info(sample_file, verbose=True)
print(metadata['name'], metadata['size'], metadata['is_file'])
print(sorted(metadata))

                          File information:                           
   C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8\sample.bin   
┌────────────┬───────────────────────────────────────────────────────┐
│ Field      │ Value                                                 │
├────────────┼───────────────────────────────────────────────────────┤
│ name       │ sample.bin                                            │
│ stem       │ sample                                                │
│ extension  │ .bin                                                  │
│ parent     │ C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8 │
│ size       │ 22                                                    │
│ human_size │ 22.00 B                                               │
│ is_file    │ True                                                  │
│ is_dir     │ False                                                 │
│ created    │ 2026-10-19 18:28:26.208091                            │
│ modi

**Another useful pattern**

In [4]:
directory_info = files.info(root)
print('Directory:', directory_info['name'])
print('Bytes below directory:', directory_info['size'])
print('Modified at:', directory_info['modified'])

Directory: klygo-files-vdjso7r8
Bytes below directory: 59
Modified at: 2026-10-19 18:28:26.224104


**Expected error and recovery**

In [5]:
try:
    files.info(root / "missing.bin")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8\missing.bin


<a id="size"></a>

### `files.size`

Calculate the size of a file or directory tree.

**Signature:** `files.size(path: 'PathInput', human: 'bool' = False) -> 'Union[int, str]'`

- `path`: Path to inspect or operate on.
- `human`: Return a human-readable size rather than an integer.

**Returns:** `int or str` — Bytes, or a formatted size when ``human=True``.

Directory size is the sum of descendant regular-file byte sizes. `human=True` returns a formatted string; use the default integer for arithmetic.

In [6]:
print(files.size(sample_file), 'bytes')
print(files.size(root, human=True))

22 bytes
59.00 B


**Another useful pattern**

In [7]:
for item in (sample_file, root):
    print(item.name, files.size(item), 'bytes /', files.size(item, human=True))

sample.bin 22 bytes / 22.00 B
klygo-files-vdjso7r8 59 bytes / 59.00 B


**Expected error and recovery**

In [8]:
try:
    files.size(root / "missing.bin")
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

FileNotFoundError: Path does not exist: C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8\missing.bin


<a id="hash"></a>

### `files.hash`

Calculate a streaming checksum for a file.

**Signature:** `files.hash(path: 'PathInput', algorithm: 'str' = 'sha256') -> 'str'`

- `path`: Path to inspect or operate on.
- `algorithm`: Algorithm accepted by ``hashlib.new``.

**Returns:** `str` — Lowercase hexadecimal checksum.

The file is read in chunks. `algorithm` is passed to `hashlib.new`; unknown names raise `ValueError`. Hashing a directory also raises `ValueError`.

In [9]:
digest = files.hash(sample_file)
print('SHA-256:', digest)
print('SHA-1:', files.hash(sample_file, algorithm='sha1'))

SHA-256: b67ded774900bab7c4b76f5ce6d9111332dc27cedac5c3c6031739ac3f24b32a
SHA-1: 074127e9034e5bc72639d9625c58df14f5db1333


**Another useful pattern**

In [10]:
print('Matching copies:', files.hash(sample_file) == files.hash(duplicate))
print('Changed file:', files.hash(different))

Matching copies: True
Changed file: 538a741c622733afe147819e939d82cc071edfbeb3eb49ed701a55975ac288b3


**Expected error and recovery**

In [11]:
try:
    files.hash(root)
except ValueError as error:
    print("ValueError:", error)

try:
    files.hash(sample_file, algorithm="not-a-hash")
except ValueError as error:
    print("ValueError:", error)

ValueError: Path must be a file to hash: C:\Users\Mine\AppData\Local\Temp\klygo-files-vdjso7r8
ValueError: unsupported hash type not-a-hash


<a id="compare"></a>

### `files.compare`

Compare two files by checksum or binary content.

**Signature:** `files.compare(path1: 'PathInput', path2: 'PathInput', by: 'str' = 'content') -> 'bool'`

- `path1`: Path to the first file.
- `path2`: Path to the second file.
- `by`: File comparison strategy.

**Returns:** `bool` — Whether both files contain identical bytes.

The default `by='content'` compares byte streams after a size check. `by='hash'` compares SHA-256 digests. Both inputs should be files.

In [12]:
print(files.compare(sample_file, duplicate, by='content'))
print(files.compare(sample_file, different, by='hash'))

True
False


**Another useful pattern**

In [13]:
for mode in ('content', 'hash'):
    print(mode, files.compare(sample_file, duplicate, by=mode))
print('Different:', files.compare(sample_file, different))

content True
hash True
Different: False


**Expected error and recovery**

In [14]:
try:
    files.compare(sample_file, duplicate, by="name")
except ValueError as error:
    print("ValueError:", error)

ValueError: Invalid compare mode: 'name'. Expected 'hash' or 'content'.


## Put it together

In [15]:
if files.compare(sample_file, duplicate):
    print('Duplicate confirmed; checksum:', files.hash(duplicate))
print('Folder bytes:', files.size(root))

Duplicate confirmed; checksum: b67ded774900bab7c4b76f5ce6d9111332dc27cedac5c3c6031739ac3f24b32a
Folder bytes: 59


## Remember

- `size(..., human=True)` is display text; use raw byte counts or `compare()` for program logic and `hash()` when you need a cryptographic digest.
- Next in `klygo.files`: continue with [Document_Files_Io.ipynb](Document_Files_Io.ipynb) ([Colab](https://colab.research.google.com/drive/1n7cKKjHpdYPEFYrVgyd8c1vGqdT3Valy?usp=sharing)) or return to [Document_Files_Overview.ipynb](Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/12PhuA8SVe_Q8jfC4f5klsgQifR6Vdf4H?usp=sharing)).
- Related Klygo modules: [`klygo.archive`](../archive/Document_Archive_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).